In [ ]:
# ============================================================
#  REQUIRED PACKAGES — run this command before starting:
#
#  pip install dash plotly pandas numpy
#
#  Then launch with:  python ecommerce_dashboard.py
#  Open browser at:  http://127.0.0.1:8050
# ============================================================

import dash
from dash import dcc, html, Input, Output
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# ─────────────────────────────────────────────────────────────────
# SYNTHETIC DATA
# ─────────────────────────────────────────────────────────────────
np.random.seed(42)

COUNTRIES = [
    "United Kingdom", "Germany", "France", "Netherlands", "Australia",
    "Switzerland", "Belgium", "Sweden", "Japan", "Norway",
    "Portugal", "Finland", "Denmark", "Austria", "Spain"
]
CATEGORIES = ["Home Furnishings", "Gift & Novelties", "Kitchen & Dining", "Stationery", "Seasonal"]

country_revenue = pd.DataFrame({
    "Country":    COUNTRIES,
    "TotalPrice": [7_400_000, 220_000, 195_000, 175_000, 140_000,
                   125_000,   110_000, 105_000,  90_000,  80_000,
                    70_000,    60_000,  55_000,  48_000,  42_000]
})

category_revenue = pd.DataFrame({
    "ProductCategory": CATEGORIES,
    "TotalPrice":      [380_000, 290_000, 210_000, 95_000, 40_000]
})

product_revenue = pd.DataFrame([
    ("P001", "REGENCY CAKESTAND 3 TIER",          "Gift & Novelties", 42_000, 1200),
    ("P002", "WHITE HANGING HEART T-LIGHT HOLDER", "Home Furnishings", 38_500, 2100),
    ("P003", "JUMBO BAG RED RETROSPOT",             "Gift & Novelties", 35_000, 1800),
    ("P004", "PARTY BUNTING",                       "Seasonal",         31_000,  900),
    ("P005", "LUNCH BAG RED RETROSPOT",             "Kitchen & Dining", 27_500, 1500),
    ("P006", "ALARM CLOCK BAKELIKE PINK",           "Home Furnishings", 25_000,  800),
    ("P007", "SET OF 3 CAKE TINS PANTRY DESIGN",   "Kitchen & Dining", 22_000,  650),
    ("P008", "RABBIT NIGHT LIGHT",                  "Home Furnishings", 20_000, 1100),
    ("P009", "MINI PAINT SET VINTAGE",              "Stationery",       18_500,  700),
    ("P010", "HAND WARMER UNION JACK",              "Seasonal",         16_000,  600),
], columns=["StockCode", "Description", "ProductCategory", "TotalRevenue", "TotalQuantity"])

monthly_revenue = pd.DataFrame({
    "Year":       2011,
    "Month":      list(range(1, 12)),
    "TotalPrice": [85_000, 92_000, 105_000, 98_000, 110_000,
                   120_000, 115_000, 130_000, 145_000, 180_000, 210_000]
})

N_CUST = 500
seg_arr = np.random.choice(
    ["High Value", "Mid Value", "Low Value"], N_CUST, p=[0.15, 0.35, 0.50]
)
orders = np.where(seg_arr == "High Value", np.random.randint(15, 50, N_CUST),
          np.where(seg_arr == "Mid Value",  np.random.randint(5,  15, N_CUST),
                                            np.random.randint(1,   5, N_CUST)))
aovs = np.where(seg_arr == "High Value", np.random.uniform(300, 800, N_CUST),
        np.where(seg_arr == "Mid Value",  np.random.uniform(100, 300, N_CUST),
                                          np.random.uniform( 20, 100, N_CUST)))
customer_value = pd.DataFrame({
    "CustomerID":    range(10000, 10000 + N_CUST),
    "Segment":       seg_arr,
    "TotalRevenue":  orders * aovs,
    "OrderCount":    orders,
    "TotalQuantity": orders * np.random.randint(2, 10, N_CUST),
    "AOV":           aovs,
})

rfm_scores = pd.DataFrame({
    "Recency":   np.random.randint(1, 365, 500),
    "Frequency": np.random.randint(1,  50, 500),
    "Monetary":  np.random.uniform(10, 5000, 500),
    "R_rank":    np.random.randint(1, 5, 500),
    "F_rank":    np.random.randint(1, 5, 500),
    "M_rank":    np.random.randint(1, 5, 500),
})
rfm_scores["RFM_Score"] = rfm_scores[["R_rank", "F_rank", "M_rank"]].sum(axis=1)

N_CANCEL = 300
cancel_dates = pd.date_range("2011-01-01", "2011-11-30", periods=N_CANCEL)
cancelled_invoices = pd.DataFrame({
    "InvoiceDate":     cancel_dates,
    "Country":         np.random.choice(COUNTRIES, N_CANCEL,
                           p=[.50,.10,.08,.07,.04,.04,.03,.03,.03,.02,.01,.01,.01,.01,.02]),
    "ProductCategory": np.random.choice(CATEGORIES, N_CANCEL),
    "CustomerID":      np.random.randint(10000, 11000, N_CANCEL),
    "Quantity":        np.random.randint(1, 20, N_CANCEL),
    "UnitPrice":       np.random.uniform(1, 50, N_CANCEL),
    "Description":     np.random.choice(product_revenue["Description"].tolist(), N_CANCEL),
    "Cancelled":       True,
})
cancelled_invoices["LossValue"] = cancelled_invoices["Quantity"] * cancelled_invoices["UnitPrice"]
cancelled_invoices["MonthNum"]  = pd.to_datetime(cancelled_invoices["InvoiceDate"]).dt.month
cancelled_invoices["MonthName"] = pd.to_datetime(cancelled_invoices["InvoiceDate"]).dt.strftime("%b")

# ─────────────────────────────────────────────────────────────────
# COLOUR PALETTE
# ─────────────────────────────────────────────────────────────────
C_PURPLE = "#6C5CE7"
C_TEAL   = "#00CEC9"
C_ORANGE = "#E17055"
C_BLUE   = "#0984e3"
C_GREEN  = "#00b894"
CARD_BG  = "#ffffff"
PAGE_BG  = "#f0eeff"

SEG_COLORS = {"High Value": C_PURPLE, "Mid Value": C_TEAL, "Low Value": C_ORANGE}
CAT_COLORS = [C_PURPLE, "#a29bfe", C_TEAL, C_ORANGE, C_GREEN]

LAYOUT_BASE = dict(
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Segoe UI, Roboto, Arial", size=12, color="#333"),
    margin=dict(l=45, r=20, t=52, b=42),
)

MONTH_LABELS = {i: lbl for i, lbl in enumerate(
    ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov"], start=1
)}

# ─────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────
def kpi_card(title, value, icon, accent):
    return html.Div([
        html.Div(icon,  style={"fontSize": "30px", "lineHeight": "1"}),
        html.Div(value, style={"fontSize": "26px", "fontWeight": "700",
                               "color": accent, "margin": "6px 0 2px"}),
        html.Div(title, style={"fontSize": "12px", "color": "#777"}),
    ], style={
        "background":   CARD_BG,
        "borderRadius": "14px",
        "padding":      "18px 22px",
        "boxShadow":    "0 2px 14px rgba(108,92,231,.12)",
        "borderLeft":   f"5px solid {accent}",
        "flex":         "1",
        "minWidth":     "155px",
    })

def card(flex=1):
    return {
        "flex":         str(flex),
        "background":   CARD_BG,
        "borderRadius": "14px",
        "padding":      "14px",
        "boxShadow":    "0 2px 12px rgba(108,92,231,.08)",
        "minWidth":     "0",
    }

def empty_fig(msg="No data for current selection"):
    """Return a blank figure with a centred message."""
    fig = go.Figure()
    fig.add_annotation(text=msg, xref="paper", yref="paper",
                       x=0.5, y=0.5, showarrow=False,
                       font=dict(size=14, color="#aaa"))
    fig.update_layout(**LAYOUT_BASE)
    return fig

def axis_style(**kw):
    base = dict(showgrid=True, gridcolor="#f0eeff", linecolor="#ddd",
                linewidth=1, zeroline=False)
    base.update(kw)
    return base

# ─────────────────────────────────────────────────────────────────
# APP LAYOUT
# ─────────────────────────────────────────────────────────────────
app = dash.Dash(__name__, suppress_callback_exceptions=True)
app.title = "E-Commerce Analytics Dashboard"

app.layout = html.Div([

    # Header
    html.Div([
        html.Div([
            html.H1("🛒 E-Commerce Analytics Dashboard",
                    style={"margin": "0", "fontSize": "22px",
                           "fontWeight": "700", "color": "white"}),
            html.P("Sales · Customer · Returns Intelligence",
                   style={"margin": "3px 0 0", "color": "rgba(255,255,255,.7)",
                          "fontSize": "13px"}),
        ]),
        html.Div("Harshil Amin & Shivangi Bulsara | 2011 Data",
                 style={"color": "rgba(255,255,255,.6)", "fontSize": "12px",
                        "alignSelf": "flex-end"}),
    ], style={
        "display": "flex", "justifyContent": "space-between", "alignItems": "center",
        "background": f"linear-gradient(135deg,{C_PURPLE},{C_TEAL})",
        "padding": "20px 32px",
    }),

    # Body
    html.Div([

        # Sidebar
        html.Div([
            html.H3("🔍 Filters", style={"color": C_PURPLE, "marginTop": "0",
                                         "fontSize": "15px", "marginBottom": "16px"}),

            html.Label("Country", style={"fontWeight": "600", "fontSize": "12px", "color": "#555"}),
            dcc.Dropdown(
                id="filter-country",
                options=[{"label": "All Countries", "value": "ALL"}] +
                        [{"label": c, "value": c} for c in sorted(COUNTRIES)],
                value="ALL", clearable=False,
                style={"marginBottom": "16px", "fontSize": "13px"},
            ),

            html.Label("Product Category", style={"fontWeight": "600", "fontSize": "12px", "color": "#555"}),
            dcc.Dropdown(
                id="filter-category",
                options=[{"label": "All Categories", "value": "ALL"}] +
                        [{"label": c, "value": c} for c in CATEGORIES],
                value="ALL", clearable=False,
                style={"marginBottom": "16px", "fontSize": "13px"},
            ),

            html.Label("Customer Segment", style={"fontWeight": "600", "fontSize": "12px", "color": "#555"}),
            dcc.Checklist(
                id="filter-segment",
                options=[{"label": f"  {s}", "value": s}
                         for s in ["High Value", "Mid Value", "Low Value"]],
                value=["High Value", "Mid Value", "Low Value"],
                labelStyle={"display": "block", "marginBottom": "7px",
                            "fontSize": "13px", "cursor": "pointer"},
                style={"marginBottom": "16px"},
            ),

            html.Hr(style={"borderColor": "#eee", "margin": "16px 0"}),

            html.Label("Month Range (2011)",
                       style={"fontWeight": "600", "fontSize": "12px", "color": "#555"}),
            html.Div(style={"height": "8px"}),
            dcc.RangeSlider(
                id="filter-month",
                min=1, max=11, step=1, value=[1, 11],
                marks={i: {"label": MONTH_LABELS[i], "style": {"fontSize": "10px"}}
                       for i in range(1, 12)},
                tooltip={"always_visible": False, "placement": "bottom"},
            ),

            html.Hr(style={"borderColor": "#eee", "margin": "20px 0 16px"}),

            html.Div([
                html.Div("💡", style={"fontSize": "18px", "marginBottom": "4px"}),
                html.P("All charts update live when you change any filter.",
                       style={"fontSize": "11px", "color": "#888", "margin": "0", "lineHeight": "1.5"}),
            ], style={"background": "#f8f6ff", "borderRadius": "10px", "padding": "12px 14px"}),

        ], style={
            "width": "220px", "flexShrink": "0",
            "background": CARD_BG, "borderRadius": "14px",
            "padding": "22px",
            "boxShadow": "0 2px 14px rgba(108,92,231,.12)",
            "alignSelf": "flex-start", "position": "sticky", "top": "20px",
        }),

        # Charts
        html.Div([
            html.Div(id="kpi-row",
                     style={"display": "flex", "gap": "14px",
                            "marginBottom": "18px", "flexWrap": "wrap"}),

            html.Div([
                html.Div(dcc.Graph(id="g-monthly",  config={"displayModeBar": False}), style=card(2)),
                html.Div(dcc.Graph(id="g-cat",      config={"displayModeBar": False}), style=card(1)),
            ], style={"display": "flex", "gap": "14px", "marginBottom": "14px"}),

            html.Div([
                html.Div(dcc.Graph(id="g-products", config={"displayModeBar": False}), style=card(2)),
                html.Div(dcc.Graph(id="g-seg-pie",  config={"displayModeBar": False}), style=card(1)),
            ], style={"display": "flex", "gap": "14px", "marginBottom": "14px"}),

            html.Div([
                html.Div(dcc.Graph(id="g-scatter",  config={"displayModeBar": False}), style=card(3)),
                html.Div(dcc.Graph(id="g-aov",      config={"displayModeBar": False}), style=card(2)),
            ], style={"display": "flex", "gap": "14px", "marginBottom": "14px"}),

            html.Div([
                html.Div(dcc.Graph(id="g-rfm",      config={"displayModeBar": False}), style=card(1)),
                html.Div(dcc.Graph(id="g-cancel",   config={"displayModeBar": False}), style=card(1)),
            ], style={"display": "flex", "gap": "14px", "marginBottom": "14px"}),

            html.Div([
                html.Div(dcc.Graph(id="g-cancel-country", config={"displayModeBar": False}), style=card(1)),
                html.Div(dcc.Graph(id="g-treemap",        config={"displayModeBar": False}), style=card(1)),
            ], style={"display": "flex", "gap": "14px"}),

        ], style={"flex": "1", "minWidth": "0"}),

    ], style={"display": "flex", "gap": "18px", "padding": "18px 20px",
              "background": PAGE_BG, "minHeight": "calc(100vh - 70px)"}),

], style={"fontFamily": "Segoe UI, Roboto, Arial, sans-serif", "background": PAGE_BG})


# ─────────────────────────────────────────────────────────────────
# MASTER CALLBACK
# ─────────────────────────────────────────────────────────────────
@app.callback(
    Output("kpi-row",          "children"),
    Output("g-monthly",        "figure"),
    Output("g-cat",            "figure"),
    Output("g-products",       "figure"),
    Output("g-seg-pie",        "figure"),
    Output("g-scatter",        "figure"),
    Output("g-aov",            "figure"),
    Output("g-rfm",            "figure"),
    Output("g-cancel",         "figure"),
    Output("g-cancel-country", "figure"),
    Output("g-treemap",        "figure"),
    Input("filter-country",  "value"),
    Input("filter-category", "value"),
    Input("filter-segment",  "value"),
    Input("filter-month",    "value"),
)
def update_all(sel_country, sel_cat, sel_segs, month_range):
    m0, m1 = month_range
    # Guard: if all segments unchecked, show all
    segs = sel_segs if sel_segs else ["High Value", "Mid Value", "Low Value"]

    # ── Slice data ──────────────────────────────────────────────
    df_monthly = monthly_revenue[monthly_revenue["Month"].between(m0, m1)].copy()

    df_cat = category_revenue.copy()
    if sel_cat != "ALL":
        df_cat = df_cat[df_cat["ProductCategory"] == sel_cat]

    df_prod = product_revenue.copy()
    if sel_cat != "ALL":
        df_prod = df_prod[df_prod["ProductCategory"] == sel_cat]

    df_cust = customer_value[customer_value["Segment"].isin(segs)].copy()

    df_country = country_revenue.copy()
    if sel_country != "ALL":
        df_country = df_country[df_country["Country"] == sel_country]

    df_cancel = cancelled_invoices.copy()
    if sel_country != "ALL":
        df_cancel = df_cancel[df_cancel["Country"] == sel_country]
    if sel_cat != "ALL":
        df_cancel = df_cancel[df_cancel["ProductCategory"] == sel_cat]
    df_cancel = df_cancel[df_cancel["MonthNum"].between(m0, m1)]

    # ── KPI Cards ───────────────────────────────────────────────
    total_rev   = df_monthly["TotalPrice"].sum()
    n_customers = len(df_cust)
    mean_aov    = df_cust["AOV"].mean() if n_customers > 0 else 0
    cancel_loss = df_cancel["LossValue"].sum()
    top_market  = (
        country_revenue.sort_values("TotalPrice", ascending=False).iloc[0]["Country"]
        if sel_country == "ALL" else sel_country
    )

    kpi_row = [
        kpi_card("Total Revenue",     f"£{total_rev:,.0f}",   "💰", C_PURPLE),
        kpi_card("Customers",         f"{n_customers:,}",     "👥", C_TEAL),
        kpi_card("Avg Order Value",   f"£{mean_aov:,.0f}",    "📦", C_BLUE),
        kpi_card("Cancellation Loss", f"£{cancel_loss:,.0f}", "⚠️", C_ORANGE),
        kpi_card("Top Market",        top_market,             "🌍", C_GREEN),
    ]

    # ── 1. Monthly Revenue Trend ─────────────────────────────────
    if df_monthly.empty:
        fig_monthly = empty_fig("No monthly data in selected range")
    else:
        df_monthly = df_monthly.copy()
        df_monthly["MonthLabel"] = df_monthly["Month"].map(MONTH_LABELS)
        fig_monthly = px.line(df_monthly, x="MonthLabel", y="TotalPrice", markers=True,
                              title="Monthly Revenue Trend (2011)",
                              labels={"TotalPrice": "Revenue (£)", "MonthLabel": "Month"})
        fig_monthly.update_traces(line=dict(color=C_PURPLE, width=3),
                                  marker=dict(size=9, color=C_PURPLE,
                                              line=dict(width=2, color="white")),
                                  fill="tozeroy", fillcolor="rgba(108,92,231,.08)")
        fig_monthly.update_layout(**LAYOUT_BASE, xaxis=axis_style(),
                                  yaxis=axis_style(tickprefix="£"), hovermode="x unified")

    # ── 2. Revenue by Category (donut) ───────────────────────────
    if df_cat.empty:
        fig_cat = empty_fig("No category data")
    else:
        # Build pull list dynamically to match actual row count
        pull_list = [0] * len(df_cat)
        fig_cat = px.pie(df_cat, names="ProductCategory", values="TotalPrice",
                         hole=0.45, title="Revenue by Category",
                         color_discrete_sequence=CAT_COLORS)
        fig_cat.update_traces(textinfo="label+percent", textposition="inside",
                              pull=pull_list)
        fig_cat.update_layout(**LAYOUT_BASE, showlegend=False)

    # ── 3. Top 10 Products ───────────────────────────────────────
    if df_prod.empty:
        fig_products = empty_fig("No products in selected category")
    else:
        top10 = df_prod.sort_values("TotalRevenue", ascending=False).head(10)
        fig_products = px.bar(top10, y="Description", x="TotalRevenue", orientation="h",
                              color="ProductCategory", title="Top 10 Products by Revenue",
                              labels={"TotalRevenue": "Revenue (£)", "Description": ""},
                              color_discrete_sequence=CAT_COLORS)
        fig_products.update_layout(
            **LAYOUT_BASE,
            yaxis=dict(autorange="reversed"),
            xaxis=axis_style(tickprefix="£"),
            showlegend=True,
            legend=dict(orientation="h", yanchor="bottom", y=1.02,
                        xanchor="right", x=1, font=dict(size=10)),
        )

    # ── 4. Customer Segment Pie ──────────────────────────────────
    if df_cust.empty:
        fig_seg = empty_fig("No customers in selected segments")
    else:
        # Robust column naming for pandas 1.x and 2.x
        vc = df_cust["Segment"].value_counts().reset_index()
        vc.columns = ["Segment", "Count"]
        fig_seg = px.pie(vc, names="Segment", values="Count", hole=0.44,
                         title="Customer Segments",
                         color="Segment", color_discrete_map=SEG_COLORS)
        fig_seg.update_traces(textinfo="label+percent", textposition="inside")
        fig_seg.update_layout(**LAYOUT_BASE, showlegend=False)

    # ── 5. Revenue vs Order Count ────────────────────────────────
    if df_cust.empty:
        fig_scatter = empty_fig("No customers in selected segments")
    else:
        fig_scatter = px.scatter(df_cust, x="OrderCount", y="TotalRevenue",
                                 color="Segment", size="TotalQuantity",
                                 hover_data=["CustomerID", "AOV"],
                                 color_discrete_map=SEG_COLORS, opacity=0.75,
                                 title="Revenue vs Order Count by Segment",
                                 labels={"TotalRevenue": "Total Revenue (£)",
                                         "OrderCount": "Order Count"})
        fig_scatter.update_layout(
            **LAYOUT_BASE,
            xaxis=axis_style(), yaxis=axis_style(tickprefix="£"),
            legend=dict(orientation="h", yanchor="bottom", y=1.02,
                        xanchor="right", x=1, font=dict(size=10)),
        )

    # ── 6. AOV Distribution ──────────────────────────────────────
    if df_cust.empty:
        fig_aov = empty_fig("No customers in selected segments")
    else:
        fig_aov = px.histogram(df_cust, x="AOV", color="Segment",
                               opacity=0.78, color_discrete_map=SEG_COLORS,
                               nbins=25, barmode="overlay",
                               title="AOV Distribution by Segment",
                               labels={"AOV": "Avg Order Value (£)"})
        fig_aov.update_layout(
            **LAYOUT_BASE,
            xaxis=axis_style(tickprefix="£"),
            yaxis=axis_style(title="Customers"),
            bargap=0.04,
            legend=dict(orientation="h", yanchor="bottom", y=1.02,
                        xanchor="right", x=1, font=dict(size=10)),
        )

    # ── 7. RFM Correlation Heatmap (not affected by filters) ─────
    rfm_cols = ["Recency", "Frequency", "Monetary", "R_rank", "F_rank", "M_rank", "RFM_Score"]
    corr = rfm_scores[rfm_cols].corr().round(2)
    fig_rfm = go.Figure(go.Heatmap(
        z=corr.values, x=corr.columns.tolist(), y=corr.index.tolist(),
        colorscale=[[0, "#ede7f6"], [0.5, "#7e57c2"], [1, "#311b92"]],
        text=corr.values, texttemplate="%{text:.2f}",
        showscale=True, colorbar=dict(thickness=12, len=0.8),
    ))
    fig_rfm.update_layout(**LAYOUT_BASE, title="RFM Correlation Heatmap",
                          margin=dict(l=80, r=20, t=52, b=60),
                          xaxis=dict(tickangle=-35))

    # ── 8. Monthly Cancellation Loss ─────────────────────────────
    mo_order = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
    if df_cancel.empty:
        fig_cancel = empty_fig("No cancellations in selected filters")
    else:
        monthly_c = (df_cancel.groupby("MonthName")["LossValue"].sum()
                              .reindex(mo_order).dropna().reset_index())
        if monthly_c.empty:
            fig_cancel = empty_fig("No cancellations in selected range")
        else:
            fig_cancel = px.line(monthly_c, x="MonthName", y="LossValue", markers=True,
                                 title="Monthly Cancellation Loss Trend",
                                 labels={"LossValue": "Loss Value (£)", "MonthName": "Month"})
            fig_cancel.update_traces(line=dict(color=C_ORANGE, width=3),
                                     marker=dict(size=8, color=C_ORANGE,
                                                 line=dict(width=2, color="white")),
                                     fill="tozeroy", fillcolor="rgba(225,112,85,.08)")
            fig_cancel.update_layout(**LAYOUT_BASE, xaxis=axis_style(),
                                     yaxis=axis_style(tickprefix="£"), hovermode="x unified")

    # ── 9. Cancellation Loss % by Country ───────────────────────
    if df_cancel.empty:
        fig_cc = empty_fig("No cancellations in selected filters")
    else:
        country_c = (df_cancel.groupby("Country")["LossValue"].sum()
                              .sort_values(ascending=False).head(10).reset_index())
        total_loss = country_c["LossValue"].sum()
        # Guard against division by zero
        country_c["Percent"] = (
            (country_c["LossValue"] / total_loss * 100).round(1)
            if total_loss > 0 else 0.0
        )
        fig_cc = px.bar(country_c, x="Percent", y="Country", orientation="h",
                        title="Top 10 Countries — Cancellation Loss %",
                        labels={"Percent": "% of Total Loss", "Country": ""},
                        color="Percent",
                        color_continuous_scale=["#f3e5f5", C_PURPLE])
        fig_cc.update_layout(**LAYOUT_BASE, yaxis=dict(autorange="reversed"),
                             coloraxis_showscale=False)

    # ── 10. Revenue by Country (treemap) ─────────────────────────
    if df_country.empty:
        fig_treemap = empty_fig("No country data")
    else:
        fig_treemap = px.treemap(df_country, path=["Country"], values="TotalPrice",
                                 color="TotalPrice",
                                 hover_data={"TotalPrice": ":,"},
                                 color_continuous_scale=["#ede7f6","#b39ddb","#7e57c2","#5e35b1","#311b92"],
                                 title="Revenue by Country (Treemap)")
        fig_treemap.update_layout(**LAYOUT_BASE, margin=dict(l=0, r=0, t=52, b=0),
                                  coloraxis_colorbar=dict(title="Revenue", thickness=12, len=0.8))

    return (
        kpi_row,
        fig_monthly, fig_cat, fig_products, fig_seg,
        fig_scatter, fig_aov, fig_rfm, fig_cancel, fig_cc, fig_treemap,
    )


# ─────────────────────────────────────────────────────────────────
# RUN
# ─────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    app.run(debug=True, port=8050)

OSError: Address 'http://127.0.0.1:8050' already in use.
    Try passing a different port to run.

: 